# 02 — Preprocessing v5 (No Per-User Norm, Deployable)

Like v4 but **without per-user normalization** so it actually works on the device.  
Keeps: magnitude features (8ch), 10 activities, 3x augmentation.

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [2]:
DATA_DIR = Path('../data')
RAW_PATH = DATA_DIR / 'raw' / 'AdamSense.csv'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_SENSOR_COLS = ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w']
ALL_SENSOR_COLS = RAW_SENSOR_COLS + ['Acc_mag', 'Gyro_mag']
NUM_FEATURES = len(ALL_SENSOR_COLS)

SAMPLING_RATE = 50
WINDOW_SIZE = 128
OVERLAP = 0.5
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP))
LABEL_THRESHOLD = 0.75
TEST_USERS = [1, 2]
DROP_ACTIVITIES = ['hand_scratching']

print(f'Window: {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLING_RATE:.2f}s)')
print(f'Features: {NUM_FEATURES} — {ALL_SENSOR_COLS}')
print(f'Dropping: {DROP_ACTIVITIES}')
print(f'Per-user normalization: NO (direct global norm for deployment)')

Window: 128 samples (2.56s)
Features: 8 — ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w', 'Acc_mag', 'Gyro_mag']
Dropping: ['hand_scratching']
Per-user normalization: NO (direct global norm for deployment)


In [3]:
df = pd.read_csv(RAW_PATH)
print(f'Total samples: {len(df):,}')

# Drop hand_scratching
df = df[~df['Activity'].isin(DROP_ACTIVITIES)].reset_index(drop=True)
print(f'After dropping {DROP_ACTIVITIES}: {len(df):,}')

# Add magnitude features on RAW data (no per-user norm)
df['Acc_mag'] = np.sqrt(df['Ax_w']**2 + df['Ay_w']**2 + df['Az_w']**2)
df['Gyro_mag'] = np.sqrt(df['Gx_w']**2 + df['Gy_w']**2 + df['Gz_w']**2)

print(f'\nActivities ({df["Activity"].nunique()}):')
for act, count in df['Activity'].value_counts().items():
    print(f'  {act:25s} {count:>8,}')

Total samples: 709,582
After dropping ['hand_scratching']: 645,092

Activities (10):
  nape_rubbing                67,859
  knuckles_cracking           67,498
  smoking                     66,937
  ear_rubbing                 66,599
  hair_pulling                65,973
  forehead_rubbing            64,604
  nail_biting                 63,457
  hand_tapping                63,220
  sitting                     59,996
  standing                    58,949


In [4]:
# Label mapping
ALL_ACTIVITIES = sorted(df['Activity'].unique().tolist())
NUM_CLASSES = len(ALL_ACTIVITIES)
LABEL_MAP = {act: i for i, act in enumerate(ALL_ACTIVITIES)}
df['label'] = df['Activity'].map(LABEL_MAP)

print(f'{NUM_CLASSES} activities:')
for act, idx in LABEL_MAP.items():
    print(f'  {idx:2d}: {act}')

10 activities:
   0: ear_rubbing
   1: forehead_rubbing
   2: hair_pulling
   3: hand_tapping
   4: knuckles_cracking
   5: nail_biting
   6: nape_rubbing
   7: sitting
   8: smoking
   9: standing


In [5]:
def extract_windows(group_df, window_size, step_size, sensor_cols, label_threshold):
    sensor_data = group_df[sensor_cols].values
    labels = group_df['label'].values
    n_samples = len(sensor_data)
    windows, window_labels = [], []
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window_lab = labels[start:end]
        counts = Counter(window_lab)
        dominant_label, dominant_count = counts.most_common(1)[0]
        if dominant_count / window_size >= label_threshold:
            windows.append(sensor_data[start:end])
            window_labels.append(dominant_label)
    return windows, window_labels


all_windows, all_labels, all_users = [], [], []
for user in sorted(df['User'].unique()):
    user_wins = 0
    for activity in ALL_ACTIVITIES:
        mask = (df['User'] == user) & (df['Activity'] == activity)
        segment = df.loc[mask]
        if len(segment) < WINDOW_SIZE:
            continue
        wins, labs = extract_windows(segment, WINDOW_SIZE, STEP_SIZE, ALL_SENSOR_COLS, LABEL_THRESHOLD)
        all_windows.extend(wins)
        all_labels.extend(labs)
        all_users.extend([user] * len(wins))
        user_wins += len(wins)
    print(f'  User {user:>2d}: {user_wins:>4d} windows')

X_all = np.array(all_windows, dtype=np.float32)
y_all = np.array(all_labels, dtype=np.int32)
users_all = np.array(all_users)

# Train/test split
test_mask = np.isin(users_all, TEST_USERS)
X_train_raw = X_all[~test_mask]
y_train = y_all[~test_mask]
X_test_raw = X_all[test_mask]
y_test = y_all[test_mask]

print(f'\nTotal: {len(X_all)} | Train: {len(X_train_raw)} | Test: {len(X_test_raw)}')

  User  2:  922 windows
  User  3:  841 windows
  User  5:  731 windows
  User  6:  851 windows
  User  7:  961 windows
  User  8:  869 windows
  User 10:  967 windows
  User 14:  948 windows
  User 15:  921 windows
  User 16: 1923 windows

Total: 9934 | Train: 9012 | Test: 922


In [6]:
# Global normalization from RAW data (this is what the device will use)
train_flat = X_train_raw.reshape(-1, NUM_FEATURES)
sensor_mean = train_flat.mean(axis=0)
sensor_std = train_flat.std(axis=0)
sensor_std[sensor_std < 1e-8] = 1.0

X_train = (X_train_raw - sensor_mean) / sensor_std
X_test = (X_test_raw - sensor_mean) / sensor_std

print('Global normalization (from raw sensor data):')
for i, col in enumerate(ALL_SENSOR_COLS):
    print(f'  {col:10s}: mean={sensor_mean[i]:>10.4f}, std={sensor_std[i]:>10.4f}')
print(f'\nThese values go directly into config.h for the device.')

Global normalization (from raw sensor data):
  Ax_w      : mean=    0.1967, std=    0.5956
  Ay_w      : mean=    0.0939, std=    0.4851
  Az_w      : mean=   -0.4514, std=    0.3930
  Gx_w      : mean=   -0.9306, std=   42.6733
  Gy_w      : mean=    1.6773, std=   20.7737
  Gz_w      : mean=    0.1284, std=   19.0177
  Acc_mag   : mean=    0.9961, std=    0.0831
  Gyro_mag  : mean=   29.9683, std=   41.4985

These values go directly into config.h for the device.


## Augmentation (3x)

In [7]:
def augment_jitter(x, sigma=0.05):
    return x + np.random.normal(0, sigma, x.shape).astype(np.float32)

def augment_scaling(x, sigma=0.1):
    return x * np.random.normal(1.0, sigma, (1, x.shape[1])).astype(np.float32)

def augment_rotation(x):
    def rand_rot(max_angle=15):
        a = np.radians(np.random.uniform(-max_angle, max_angle, 3))
        cx, cy, cz = np.cos(a); sx, sy, sz = np.sin(a)
        Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
        Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
        Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
        return Rz @ Ry @ Rx
    result = x.copy()
    result[:, :3] = (rand_rot() @ result[:, :3].T).T
    result[:, 3:6] = (rand_rot() @ result[:, 3:6].T).T
    result[:, 6] = np.sqrt(result[:, 0]**2 + result[:, 1]**2 + result[:, 2]**2)
    result[:, 7] = np.sqrt(result[:, 3]**2 + result[:, 4]**2 + result[:, 5]**2)
    return result.astype(np.float32)

def augment_time_warp(x, sigma=0.2, num_knots=4):
    from scipy.interpolate import CubicSpline
    orig = np.arange(x.shape[0])
    kp = np.linspace(0, x.shape[0]-1, num_knots+2)
    kv = kp + np.random.normal(0, sigma, len(kp)).cumsum()
    kv = np.clip(kv, 0, x.shape[0]-1); kv[0]=0; kv[-1]=x.shape[0]-1
    warped = np.clip(CubicSpline(kp, kv)(orig), 0, x.shape[0]-1)
    result = np.zeros_like(x)
    for c in range(x.shape[1]):
        result[:, c] = np.interp(warped, orig, x[:, c])
    return result

def augment_magnitude_warp(x, sigma=0.2, num_knots=4):
    from scipy.interpolate import CubicSpline
    orig = np.arange(x.shape[0])
    kp = np.linspace(0, x.shape[0]-1, num_knots+2)
    warp = CubicSpline(kp, np.random.normal(1.0, sigma, len(kp)))(orig).astype(np.float32)
    return x * warp[:, np.newaxis]

def apply_augmentation(x, p=0.5):
    x = x.copy()
    if np.random.random() < p: x = augment_jitter(x)
    if np.random.random() < p: x = augment_scaling(x)
    if np.random.random() < p: x = augment_rotation(x)
    if np.random.random() < 0.3: x = augment_time_warp(x)
    if np.random.random() < 0.3: x = augment_magnitude_warp(x)
    return x

NUM_AUG_COPIES = 2
X_aug_list = [X_train]
y_aug_list = [y_train]
for i in range(NUM_AUG_COPIES):
    print(f'Augmenting copy {i+1}/{NUM_AUG_COPIES}...')
    X_aug_list.append(np.array([apply_augmentation(X_train[j]) for j in range(len(X_train))], dtype=np.float32))
    y_aug_list.append(y_train.copy())

X_train_aug = np.concatenate(X_aug_list)
y_train_aug = np.concatenate(y_aug_list)
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print(f'\nTrain: {X_train.shape[0]} -> {X_train_aug.shape[0]} (3x)')
print(f'Test: {X_test.shape[0]}')

Augmenting copy 1/2...
Augmenting copy 2/2...

Train: 9012 -> 27036 (3x)
Test: 922


## Save

In [8]:
np.save(PROCESSED_DIR / 'X_train.npy', X_train_aug)
np.save(PROCESSED_DIR / 'y_train.npy', y_train_aug)
np.save(PROCESSED_DIR / 'X_test.npy', X_test)
np.save(PROCESSED_DIR / 'y_test.npy', y_test)
np.save(PROCESSED_DIR / 'X_train_original.npy', X_train)
np.save(PROCESSED_DIR / 'y_train_original.npy', y_train)

preprocessing_info = {
    'sensor_columns': ALL_SENSOR_COLS,
    'raw_sensor_columns': RAW_SENSOR_COLS,
    'num_features': NUM_FEATURES,
    'num_classes': NUM_CLASSES,
    'activities': ALL_ACTIVITIES,
    'label_map': LABEL_MAP,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'overlap': OVERLAP,
    'sampling_rate': SAMPLING_RATE,
    'sensor_mean': sensor_mean,
    'sensor_std': sensor_std,
    'test_users': TEST_USERS,
    'train_samples_original': len(X_train),
    'train_samples_augmented': len(X_train_aug),
    'test_samples': len(X_test),
    'dropped_activities': DROP_ACTIVITIES,
    'per_user_normalization': False,
    'magnitude_features': True,
}

with open(PROCESSED_DIR / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print(f'=== Preprocessing v5 Complete ===')
print(f'Activities: {NUM_CLASSES} (dropped: {DROP_ACTIVITIES})')
print(f'Features: {NUM_FEATURES} (raw + magnitude)')
print(f'Per-user normalization: NO')
print(f'Train: {len(X_train_aug)} | Test: {len(X_test)}')
print(f'\nLabel mapping:')
for act, idx in LABEL_MAP.items():
    print(f'  {idx:2d}: {act}')

=== Preprocessing v5 Complete ===
Activities: 10 (dropped: ['hand_scratching'])
Features: 8 (raw + magnitude)
Per-user normalization: NO
Train: 27036 | Test: 922

Label mapping:
   0: ear_rubbing
   1: forehead_rubbing
   2: hair_pulling
   3: hand_tapping
   4: knuckles_cracking
   5: nail_biting
   6: nape_rubbing
   7: sitting
   8: smoking
   9: standing
